# IND320 – Compulsory work 1: Norwegian water reservoirs

## Links
- **GitHub repository:** https://github.com/maryanahmed11011-pixel/IND320.prosjekt.maryan
- **Streamlit app:** https://ind320prosjektmaryan-lmvbjxajmrnrqkykq8ivne.streamlit.app/
- **Screencast:** FYLL INN LENKE


## AI usage
AI tools were used throughout this compulsory work, as encouraged in the course. I used Claude (Anthropic) as a step-by-step assistant for setting up a Python virtual environment in VS Code, configuring Git and connecting the local folder to GitHub, and troubleshooting errors in the terminal. Claude also suggested code for reading and renaming the data, the plots in this notebook and the four Streamlit pages, and helped fix a layout issue in the interactive plot. An AI tool was also used to generate an initial task outline, and Claude helped draft this log based on my own work process. I have run, tested and checked all code myself, both locally and in the deployed Streamlit app, and I am responsible for the final content.

## Work log (300–500 words)
## Work log
The goal of this compulsory work was to build the foundation for the IND320 project: a Jupyter Notebook for exploring the data, and a Streamlit app deployed online from a public GitHub repository.

Setting up the development environment took more time than expected. My computer had several Python installations, and my first package installation ended up in the wrong one. I solved this by creating a virtual environment (.venv) inside the project folder, activating it in the VS Code terminal and installing streamlit, pandas, matplotlib and jupyter there. I learned to check that the terminal shows (.venv) before running commands, and to use python -m pip to be sure packages end up in the active environment. I also added a .gitignore so that the environment and cache files are not uploaded to GitHub.

Next, I initialised Git in the project folder, connected it to my GitHub repository and configured my name and e-mail. A missing quotation mark in one command stored a wrong e-mail address, which I fixed with --replace-all. This taught me to read terminal messages carefully.

The data file, reservoirs.csv, is located in D2Dbook/data in the course repository. When inspecting it, I found that the data is in a long format: each row is one week for one area, covering 1995 to 2026, and the file contains the whole of Norway (NO), the five price areas (EL) and water regions (VASS). To get one clear time series, I chose to use the national total and sorted the rows by date. I renamed all headers to English, for example fyllingsgrad to filling_degree and kapasitet_TWh to capacity_TWh, and converted the date column to real dates.

I plotted each measurement separately and then all of them together. Since the columns have very different scales (TWh values versus shares between 0 and 1), I used min-max normalisation to put them on a common 0-1 scale.

The Streamlit app has four pages. A shared utils.py reads the data with @st.cache_data so the file is only loaded once. The data table page shows one row per column with a LineChartColumn for the first month, and the plot page uses st.selectbox and st.select_slider with the first month as default. After deployment, I noticed that the plot changed size when moving the slider. This was caused by automatic cropping of the image, and I fixed it with a fixed layout, fixed axis limits and by displaying the figure with st.image.

Overall, I learned a lot about development environments, Git and the connection between local analysis and an online app.

## 1. Import libraries

In [ ]:
# pandas brukes til å lese og behandle tabelldata, matplotlib til plotting
import pandas as pd
import matplotlib.pyplot as plt

## 2. Read the data and rename headers to English

In [ ]:
# Les CSV-filen (ligger i data-mappen i prosjektet)
df_raw = pd.read_csv("data/reservoirs.csv")

# Oversett kolonnenavnene fra norsk til engelske, forklarende navn
rename_map = {
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_degree",
    "kapasitet_TWh": "capacity_TWh",
    "fylling_TWh": "filling_TWh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "filling_degree_prev_week",
    "endring_fyllingsgrad": "filling_degree_change",
}
df = df_raw.rename(columns=rename_map)

# Gjør datokolonnen om til ekte datoer, så plottene får riktig tidsakse
df["date"] = pd.to_datetime(df["date"])
df.columns.tolist()

## 3. Inspect the contents

In [ ]:
# Størrelse på datasettet (rader, kolonner) og datatyper
print("Shape:", df.shape)
df.info()

In [ ]:
# Filen inneholder flere typer områder:
#   NO   = hele Norge (area_number 0)
#   EL   = strømområdene NO1-NO5 (area_number 1-5)
#   VASS = vassdragsregioner
print(df["area_type"].value_counts())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

In [ ]:
# Vi bruker totalen for hele Norge, slik at vi får én ukentlig tidsserie.
# Radene i filen står ikke i datorekkefølge, så vi sorterer etter dato.
norway = df[df["area_type"] == "NO"].sort_values("date").reset_index(drop=True)

# Måleverdiene vi vil se nærmere på
measure_cols = ["filling_degree", "capacity_TWh", "filling_TWh",
                "filling_degree_prev_week", "filling_degree_change"]

# Vis de første radene og et statistisk sammendrag
display(norway.head(10))
display(norway[measure_cols].describe().round(3))

## 4. Plot each column separately

In [ ]:
# Ett delplott per måleverdi, med felles tidsakse
fig, axes = plt.subplots(len(measure_cols), 1, figsize=(12, 2.6 * len(measure_cols)), sharex=True)

for ax, col in zip(axes, measure_cols):
    ax.plot(norway["date"], norway[col], linewidth=0.8)
    ax.set_title(col)
    ax.grid(True, alpha=0.4)

axes[-1].set_xlabel("Date")
fig.suptitle("Norwegian reservoirs (national total, weekly)")
plt.tight_layout()
plt.show()

## 5. Plot all columns together
Kolonnene har ulike skalaer: kapasitet og fylling måles i TWh (titalls TWh),
mens fyllingsgrad er en andel mellom 0 og 1, og endringen er et lite tall rundt 0.
For å kunne sammenligne formen på kurvene i ett plott, min-max-normaliseres
hver kolonne til skalaen 0–1.

In [ ]:
# Min-max-normalisering: (verdi - min) / (max - min) gir 0-1 for hver kolonne
values = norway[measure_cols]
spread = (values.max() - values.min()).replace(0, 1)  # unngå deling på 0 hvis en kolonne er konstant
normalised = (values - values.min()) / spread

fig, ax = plt.subplots(figsize=(12, 5))
for col in measure_cols:
    ax.plot(norway["date"], normalised[col], linewidth=0.8, label=col)

ax.set_title("All columns together (min-max normalised)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised value (0-1)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Samme data for bare ett år, der sesongvariasjonen er lettere å se
one_year = norway[norway["date"].dt.year == 2023]
norm_year = (one_year[measure_cols] - values.min()) / spread

fig, ax = plt.subplots(figsize=(12, 4))
for col in measure_cols:
    ax.plot(one_year["date"], norm_year[col], marker="o", markersize=3, label=col)
ax.set_title("All columns together, 2023 (normalised)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised value (0-1)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()